## Step 1 — Member group: CineClub members active on Jan 1, 2026

**What this does**
Builds the list of every CineClub member who was active on Jan 1, 2026, and labels each one once:
- **Bucket**, from their last 2025 event:
  - *First-time*: first-ever sign-up in 2025
  - *Returning*: re-joined in 2025
  - *Upgrade*: completed a monthly → annual upgrade in 2025
  - *Existing*: joined before 2025
- **Plan**: their plan on Dec 31, 2025 (latest snapshot if missing). The plan stays fixed for the whole analysis; switching plans later still counts as staying.

**So what**
This is the "test" group. Every later step (matching, monthly KPIs, churn) starts from this list, so if these counts are wrong, everything after is wrong. That's why the check cell must pass before moving on.

In [0]:
from pyspark.sql import functions as F, Window as W

EVENTS     = "cpx_dataanalytics_gold.marketing.mkt_subscription_events"
SNAP       = "cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot"
FEAT       = "data_exploration_hub.marketing_hub.propensity_features"
AS_OF      = "2026-01-01"   # start of analysis (feature snapshot)
PRIOR_YEAR = 2025           # year whose sign-ups define the buckets
SNAP_ID    = "20251231"     # subscriber snapshot for the plan

In [0]:
ev = spark.table(EVENTS)

# Upgrade completions: monthly term ends because of a move to annual
upg = (ev.filter((F.col("event_type") == "subscription_cancelled")
                 & (F.col("plan_id") == "monthly1") & (F.col("changed") == True))
         .select("cc_user_profile_id", "reporting_datetime"))

# Real sign-ups (annual "sign-ups" that are really upgrades removed), numbered per person
upg_month = upg.select("cc_user_profile_id",
                       F.date_trunc("MONTH", "reporting_datetime").alias("m"))
created = (ev.filter((F.col("event_type") == "subscription_created")
                     & F.col("cc_user_profile_id").isNotNull())
             .withColumn("m", F.date_trunc("MONTH", "reporting_datetime"))
             .join(upg_month, ["cc_user_profile_id", "m"], "left_anti")
             .unionByName(ev.filter((F.col("event_type") == "subscription_created")
                                    & F.col("cc_user_profile_id").isNotNull()
                                    & (F.col("plan_id") != "annual1"))
                            .withColumn("m", F.date_trunc("MONTH", "reporting_datetime"))
                            .join(upg_month, ["cc_user_profile_id", "m"], "left_semi"))
             .select("cc_user_profile_id", "reporting_datetime")
             .dropDuplicates(["cc_user_profile_id", "reporting_datetime"])   # NEW: tie rule 1
             .withColumn("nth_signup", F.row_number().over(
                 W.partitionBy("cc_user_profile_id").orderBy("reporting_datetime"))))

# Every PRIOR_YEAR event with its bucket
events_yr = (created.filter(F.year("reporting_datetime") == PRIOR_YEAR)
               .withColumn("bucket", F.when(F.col("nth_signup") == 1, "First-time")
                                      .otherwise("Returning"))
               .drop("nth_signup")
               .unionByName(upg.filter(F.year("reporting_datetime") == PRIOR_YEAR)
                               .withColumn("bucket", F.lit("Upgrade"))))

# One row per CDE_ID from a subscriber snapshot
def plan_lookup(snapshot_id):
    s = (spark.table(SNAP).filter(F.col("snapshot_date_id") == snapshot_id)
           .select("cc_user_profile_id", F.col("cde_id").cast("int").alias("CDE_ID"), "plan_id"))
    return (s.withColumn("rn", F.row_number().over(W.partitionBy("CDE_ID").orderBy("plan_id")))
             .filter("rn = 1").drop("rn"))

latest_id   = spark.table(SNAP).agg(F.max("snapshot_date_id")).first()[0]
snap_dec    = plan_lookup(SNAP_ID)
snap_latest = plan_lookup(latest_id)

# Bucket = last PRIOR_YEAR event (upgrades count only if the person is on annual1 at Dec 31)
last_state = (events_yr.join(snap_dec, "cc_user_profile_id")
                .filter(~((F.col("bucket") == "Upgrade") & (F.col("plan_id") != "annual1")))
                .withColumn("rn", F.row_number().over(
                    W.partitionBy("CDE_ID").orderBy(
                        F.col("reporting_datetime").desc(),
                        F.when(F.col("bucket") == "Upgrade", 0).otherwise(1))))   # NEW: tie rule 2
                .filter("rn = 1").select("CDE_ID", "bucket"))

# Active on AS_OF + plan (Dec 31 first, latest snapshot fallback) + bucket
active = (spark.table(FEAT)
            .filter((F.col("as_of_date") == AS_OF) & (F.col("cc_status") == "Active"))
            .select("CDE_ID"))

all_active = (active
    .join(snap_dec.select("CDE_ID", F.col("plan_id").alias("plan_dec")), "CDE_ID", "left")
    .join(snap_latest.select("CDE_ID", F.col("plan_id").alias("plan_latest")), "CDE_ID", "left")
    .join(last_state, "CDE_ID", "left")
    .withColumn("plan_id", F.coalesce("plan_dec", "plan_latest"))
    .withColumn("plan_source", F.when(F.col("plan_dec").isNotNull(), "Dec 31")
                                .when(F.col("plan_latest").isNotNull(), "Latest"))
    .withColumn("bucket", F.coalesce("bucket", F.lit("Existing")))
    .select("CDE_ID", "bucket", "plan_id", "plan_source"))

excluded = all_active.filter(F.col("plan_id").isNull())      # CDE_ID not linked to snapshot
members  = all_active.filter(F.col("plan_id").isNotNull()).cache()

In [0]:
expected = {
    ("Existing", "annual1"): 31034, ("Existing", "monthly1"): 75528, ("Existing", "gift-plan"): 321,
    ("First-time", "annual1"): 18160, ("First-time", "monthly1"): 37907, ("First-time", "gift-plan"): 3227,
    ("Returning", "annual1"): 9832, ("Returning", "monthly1"): 40122, ("Returning", "gift-plan"): 1386,
    ("Upgrade", "annual1"): 9965,
}
actual = {(r["bucket"], r["plan_id"]): r["count"]
          for r in members.groupBy("bucket", "plan_id").count().collect()}

display(members.groupBy("bucket").pivot("plan_id").count().orderBy("bucket"))
print(f"Members: {members.count():,} | Unique CDE_IDs: {members.select('CDE_ID').distinct().count():,}")
print(f"Excluded (no plan): {excluded.count():,}")
print("Plan source:", {r['plan_source']: r['count'] for r in members.groupBy('plan_source').count().collect()})

diffs = {k: (expected.get(k), actual.get(k)) for k in set(expected) | set(actual)
         if expected.get(k) != actual.get(k)}
print("PASS" if not diffs else f"DIFFERENCES (expected, actual): {diffs}")

## Step 2 — Non-member pool

**What this does**
Builds the pool of possible lookalikes: people who had never joined CineClub as of Jan 1, 2026 **and** didn't join at any point through Aug 2026 (the last month of data). People who joined during 2026 are left out; they're Part 2.

**So what**
The pool is ~7.7M people, but on average they go to the movies far less than members (about 1.4 vs 9.6 visits a year). Comparing members to everyone would make CineClub look much better than it is. That's why Step 3 picks only the non-members who **behave like** each member. We keep the full pool, not a sample, so heavy movie-goers still find lookalikes.

In [0]:
POOL_END = "2026-08-01"   # last loaded snapshot; "never joined" is checked through here

feat = spark.table(FEAT)

joined_2026 = (feat.filter(F.col("as_of_date").between("2026-02-01", POOL_END)
                           & (F.col("cc_status") != "Never"))
                   .select("CDE_ID").distinct())

pool = (feat.filter((F.col("as_of_date") == AS_OF) & (F.col("cc_status") == "Never"))
            .select("CDE_ID")
            .join(joined_2026, "CDE_ID", "left_anti")
            .cache())

In [0]:
n_pool    = pool.count()
n_overlap = pool.join(members, "CDE_ID").count()

print(f"Pool: {n_pool:,} | Overlap with members: {n_overlap}")
print("PASS" if (n_pool == 7745758 and n_overlap == 0) else "CHECK FAILED")

## Step 3 — Direct match (Section 1)

**What this does**
Puts every member and every non-member in the pool into a "box" based on 4 traits on Jan 1, 2026:
- **RFM segment** (`exh_rfm_segment`): official segment names, with cut-offs recalculated each quarter from percentiles
- **LBE active**: any Rec Room / Playdium visit in the last 12 months
- **Pricing tier** of their home theatre
- **Age band**

Each member is then compared only with non-members **in the same box**.

**So what**
Direct matching is simple and easy to explain ("same segment, same Rec Room habit, same pricing area, same age"). But boxes are coarse: inside "Champions", a member may go 20 times a year and their lookalike 8. The balance check at the end shows how much difference is left. That leftover gap is why Section 2 (propensity matching) exists.

In [0]:
BOX = ["exh_rfm_segment", "lbe_active", "home_tier", "demo_age_band"]

feat_asof = (spark.table(FEAT).filter(F.col("as_of_date") == AS_OF)
               .select("CDE_ID", "exh_rfm_segment",
                       (F.col("lbe_frequency") > 0).cast("int").alias("lbe_active"),
                       F.coalesce("home_tier", F.lit("Unknown")).alias("home_tier"),
                       F.coalesce("demo_age_band", F.lit("Unknown")).alias("demo_age_band")))

members_box = members.join(feat_asof, "CDE_ID").cache()
pool_box    = pool.join(feat_asof, "CDE_ID").cache()

In [0]:
LOOKALIKES = 5

boxes = (members_box.groupBy(BOX).agg(F.count("*").alias("members"))
           .join(pool_box.groupBy(BOX).agg(F.count("*").alias("pool")), BOX, "left")
           .fillna(0, ["pool"])
           .withColumn("ok_with_reuse", F.when(F.col("pool") >= LOOKALIKES, F.col("members")).otherwise(0))
           .withColumn("ok_no_reuse",   F.least("members", F.floor(F.col("pool") / LOOKALIKES))))

summary = (boxes.groupBy("exh_rfm_segment")
             .agg(F.sum("members").alias("members"), F.sum("pool").alias("pool_in_member_boxes"),
                  F.sum("ok_with_reuse").alias("matchable_with_reuse"),
                  F.sum("ok_no_reuse").alias("matchable_no_reuse"))
             .withColumn("pct_with_reuse", F.round(100 * F.col("matchable_with_reuse") / F.col("members"), 1))
             .withColumn("pct_no_reuse",   F.round(100 * F.col("matchable_no_reuse") / F.col("members"), 1))
             .orderBy(F.desc("members")))
display(summary)

tot = boxes.agg(F.sum("members"), F.sum("ok_with_reuse"), F.sum("ok_no_reuse")).first()
print(f"Members: {tot[0]:,} | with reuse: {100*tot[1]/tot[0]:.1f}% | no reuse: {100*tot[2]/tot[0]:.1f}%")
print(f"Members boxed: {members_box.count():,} (expect 227,482) | Pool boxed: {pool_box.count():,} (expect 7,745,758)")

## Step 4 — Propensity score (Section 2)

**What this does**
For each plan (annual, monthly, gift), a model learns what members of that plan look like compared with non-members, using ~50 behaviour and profile features from Jan 1, 2026. It then gives **everyone** a score: "how much does this person look like a member of this plan?"

People with the same score look alike **on all features at once**, not just the 4 box traits used in direct matching.

**So what**
Matching on this score fixes the coarseness of direct matching: a 20-visit member gets matched with a non-member who also looks like a 20-visit member. The overlap table at the end shows whether enough non-members look like members to find 5 lookalikes each.

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

# Score model features (exact duplicates removed)
NUM = ["exh_recency_days", "exh_frequency", "exh_monetary", "exh_net_revenue", "exh_movie_visits",
       "exh_momentum", "exh_concession_incidence", "exh_avg_group_size",
       "exh_share_premium_any", "exh_share_imax", "exh_share_avx", "exh_share_vip",
       "exh_share_dbox", "exh_share_4dx", "exh_share_screenx", "exh_share_3d", "exh_share_altprog",
       "exh_dow_share_mon", "exh_dow_share_tue", "exh_dow_share_wed", "exh_dow_share_thu",
       "exh_dow_share_fri", "exh_dow_share_sat",
       "lbe_recency_days", "lbe_frequency", "lbe_monetary", "lbe_food_spend", "lbe_games_spend", "lbe_momentum",
       "lbe_dow_share_mon", "lbe_dow_share_tue", "lbe_dow_share_wed", "lbe_dow_share_thu",
       "lbe_dow_share_fri", "lbe_dow_share_sat"]
# Balance check uses ALL features, including the 4 dropped from the model
BALANCE = NUM + ["exh_concession_only_visits", "exh_dow_share_sun", "lbe_dow_share_sun", "exh_share_regular"]
CAT = ["demo_gender", "demo_age_band", "demo_language", "demo_province",
       "home_tier", "home_market", "home_source"]
TRAIN_POOL = 200_000        # pool sample used for TRAINING only; everyone is scored
SEED       = 42
SCORES_TBL = "data_exploration_hub.marketing_hub.t1_propensity_scores"

base = (members.select("CDE_ID", "plan_id")
          .unionByName(pool.withColumn("plan_id", F.lit(None).cast("string")))
          .join(spark.table(FEAT).filter(F.col("as_of_date") == AS_OF).select("CDE_ID", *BALANCE, *CAT), "CDE_ID")
          .fillna("Unknown", CAT)
          .withColumn("label", F.col("plan_id").isNotNull().cast("double")))

train_raw = (base.filter("label = 1")
               .unionByName(base.filter("label = 0").sample(False, TRAIN_POOL / 7745758, SEED))
               .cache())

# Cap extremes at p99.9, cut-offs from the training sample (locked outlier rule: cap at model stage)
q99  = train_raw.approxQuantile(BALANCE, [0.999], 0.001)
caps = [F.least(F.col(c).cast("double"), F.lit(q99[i][0])).alias(c) for i, c in enumerate(BALANCE)]
cap  = lambda df: df.select("CDE_ID", "plan_id", "label", *CAT, *caps)

print(f"Training rows: {train_raw.count():,} (members 227,482 + ~200K pool)")

In [0]:
%pip install -U "jax[cpu]"

In [0]:
import numpy as np, pandas as pd, jax, jax.numpy as jnp
from jax.scipy.optimize import minimize
from functools import reduce
from sklearn.metrics import roc_auc_score
jax.config.update("jax_enable_x64", True)

L2 = 1e-3   # small ridge penalty: keeps collinear features stable

# 1. Training sample to the driver; one-hot the categoricals
pdf  = cap(train_raw).toPandas()
dums = pd.get_dummies(pdf[CAT].astype(str), prefix_sep="==", drop_first=True, dtype=float)
cols = NUM + list(dums.columns)
X    = np.hstack([pdf[NUM].astype(float).values, dums.values])
y    = pdf["label"].values.astype(float)

# 2. Standardize
mu, sd = X.mean(0), X.std(0)
sd[sd == 0] = 1.0
Xs = jnp.asarray((X - mu) / sd)
yj = jnp.asarray(y)

# 3. Logistic regression: mean log-loss + L2, solved with BFGS
def loss(theta):
    z = Xs @ theta[1:] + theta[0]
    return jnp.mean(jnp.logaddexp(0.0, z) - yj * z) + 0.5 * L2 * jnp.sum(theta[1:] ** 2)

res   = minimize(jax.jit(loss), jnp.zeros(Xs.shape[1] + 1), method="BFGS")
b, w  = float(res.x[0]), np.asarray(res.x[1:])
print(f"Converged: {bool(res.success)} | AUC (training sample): {roc_auc_score(y, np.asarray(Xs @ w + b)):.3f}")

# 4. Score everyone in Spark: logit = b + sum(w * (x - mu) / sd)
def term(i, c):
    x = (F.when(F.col(c.split("==")[0]) == c.split("==")[1], 1.0).otherwise(0.0)
         if "==" in c else F.col(c))
    return (x - float(mu[i])) * float(w[i] / sd[i])

logit_expr = reduce(lambda a, t: a + t, [term(i, c) for i, c in enumerate(cols)], F.lit(b))

(cap(base)
   .select("CDE_ID", "plan_id", F.lit("all").alias("score_plan"), "label", logit_expr.alias("logit"))
   .write.mode("overwrite").saveAsTable(SCORES_TBL))

scores = spark.table(SCORES_TBL).cache()
print(f"Scored: {scores.count():,} (expect 7,973,240)")

In [0]:
# Caliper = 0.2 x SD of the logit (per plan). Bin scores into caliper-wide bins and check
# whether each members' bin has >= 5 non-members (with reuse) or 5 unused ones (no reuse).
sd = {r["score_plan"]: r["sd"] for r in scores.groupBy("score_plan").agg(F.stddev("logit").alias("sd")).collect()}
cal = F.create_map(*[x for p, s in sd.items() for x in (F.lit(p), F.lit(0.2 * s))])

bins = (scores.withColumn("bin", F.floor(F.col("logit") / cal[F.col("score_plan")]))
          .groupBy("score_plan", "bin")
          .agg(F.sum("label").alias("members"), F.sum(1 - F.col("label")).alias("pool"))
          .filter("members > 0")
          .withColumn("ok_with_reuse", F.when(F.col("pool") >= LOOKALIKES, F.col("members")).otherwise(0))
          .withColumn("ok_no_reuse",   F.least("members", F.floor(F.col("pool") / LOOKALIKES))))

display(bins.groupBy("score_plan")
            .agg(F.sum("members").cast("long").alias("members"),
                 F.round(100 * F.sum("ok_with_reuse") / F.sum("members"), 1).alias("pct_with_reuse"),
                 F.round(100 * F.sum("ok_no_reuse") / F.sum("members"), 1).alias("pct_no_reuse"))
            .orderBy("score_plan"))
print("Caliper (0.2 x SD logit):", {p: round(0.2 * s, 3) for p, s in sd.items()})

## Step 5 — Propensity match (Section 2)

**What this does**
For each member, finds the **5 non-members with the closest propensity score** (closest overall look-alike profile). A pair only counts if the scores are within the **caliper**: 0.2 × the spread of the scores, a standard "close enough" rule. Members without 5 lookalikes that close are left out and reported. A non-member **can** serve as lookalike for more than one member (matching with replacement), because heavy movie-goers who never joined CineClub are rare.

**So what**
If matching worked, the balance check shows members and their lookalikes look the same on **every** feature (standardized difference under 0.1), not just the 4 traits used in direct matching. Only then can later differences in behaviour be credited to CineClub rather than to members simply being different people.

In [0]:
import numpy as np
K = 5
PAIRS_TBL = "data_exploration_hub.marketing_hub.t1_matched_pairs"
EXACT = ["exh_f_score", "exh_r_score", "exh_m_score", "demo_age_band"]

keys = (spark.table(FEAT).filter(F.col("as_of_date") == AS_OF)
          .select("CDE_ID", *[F.coalesce(F.col(c).cast("string"), F.lit("Unknown")).alias(c) for c in EXACT]))
sc  = scores.select("CDE_ID", "label", "logit").join(keys, "CDE_ID").toPandas()
cal = 0.2 * sc["logit"].std()

out, drop = [], {"no_pool_in_group": 0, "outside_caliper": 0}
for _, g in sc.groupby(EXACT, sort=False):
    mem = g[g["label"] == 1]
    if mem.empty:
        continue
    pl = g[g["label"] == 0].sort_values("logit")
    if len(pl) < K:
        drop["no_pool_in_group"] += len(mem); continue
    pv, mv = pl["logit"].values, mem["logit"].values
    pos  = np.searchsorted(pv, mv)[:, None] + np.arange(-K, K)[None, :]
    oob  = (pos < 0) | (pos >= len(pv))
    cand = np.clip(pos, 0, len(pv) - 1)
    dist = np.where(oob, np.inf, np.abs(pv[cand] - mv[:, None]))
    top   = np.argsort(dist, axis=1)[:, :K]
    best  = np.take_along_axis(cand, top, 1)
    bestd = np.take_along_axis(dist, top, 1)
    ok    = (bestd <= cal).all(axis=1)
    drop["outside_caliper"] += int((~ok).sum())
    out.append(pd.DataFrame({
        "member_id":    np.repeat(mem["CDE_ID"].values[ok], K),
        "lookalike_id": pl["CDE_ID"].values[best[ok].ravel()],
        "score_gap":    bestd[ok].ravel()}))

pairs_pdf = pd.concat(out, ignore_index=True)
spark.createDataFrame(pairs_pdf).write.mode("overwrite").saveAsTable(PAIRS_TBL)
pairs = spark.table(PAIRS_TBL).cache()

n_mem = int((sc["label"] == 1).sum()); n_ok = pairs_pdf["member_id"].nunique()
print(f"Caliper: {cal:.4f} | Groups: {sc.groupby(EXACT).ngroups:,} | "
      f"Members matched: {n_ok:,} of {n_mem:,} ({100*n_ok/n_mem:.1f}%) | Dropped: {drop}")

In [0]:
uses = pairs.groupBy("lookalike_id").count()
u = uses.agg(F.count("*").alias("distinct"), F.max("count").alias("max_uses"),
             F.expr("percentile_approx(count, 0.5)").alias("median_uses"),
             F.expr("percentile_approx(count, 0.99)").alias("p99_uses")).first()
n_pairs = pairs.count()
print(f"Pairs: {n_pairs:,} | Distinct lookalikes: {u['distinct']:,} | "
      f"Uses per lookalike: median {u['median_uses']}, p99 {u['p99_uses']}, max {u['max_uses']}")
print(f"Avg score gap: {pairs.agg(F.avg('score_gap')).first()[0]:.4f} (caliper {cal:.4f})")

dropped = members.join(pairs.select(F.col("member_id").alias("CDE_ID")).distinct(), "CDE_ID", "left_anti")
display(dropped.groupBy("plan_id", "bucket").count().orderBy(F.desc("count")))

In [0]:
BAL_CAT = CAT + ["exh_rfm_segment", "exh_momentum_band", "exh_r_score", "exh_f_score", "exh_m_score"]
extra = (spark.table(FEAT).filter(F.col("as_of_date") == AS_OF)
           .select("CDE_ID", "exh_rfm_segment", "exh_momentum_band",
                   *[F.col(c).cast("string").alias(c) for c in ["exh_r_score", "exh_f_score", "exh_m_score"]]))
fc = cap(base).select("CDE_ID", *BALANCE, *CAT).join(extra, "CDE_ID")

groups = {
    "members_all":  fc.join(members.select("CDE_ID"), "CDE_ID"),
    "pool_all":     fc.join(pool, "CDE_ID"),
    "members_kept": fc.join(pairs.select(F.col("member_id").alias("CDE_ID")).distinct(), "CDE_ID"),
    "lookalikes":   fc.join(pairs.select(F.col("lookalike_id").alias("CDE_ID")), "CDE_ID"),  # keeps reuse as weight
}

def num_stats(df):
    r = df.agg(*[F.mean(c).alias(f"m_{c}") for c in BALANCE], *[F.var_pop(c).alias(f"v_{c}") for c in BALANCE]).first()
    return {c: (r[f"m_{c}"], r[f"v_{c}"]) for c in BALANCE}

def cat_stats(df):
    n = df.count(); out = {}
    for c in BAL_CAT:
        for row in df.groupBy(c).count().collect():
            p = row["count"] / n
            out[f"{c}={row[c]}"] = (p, p * (1 - p))
    return out

st = {g: {**num_stats(d), **cat_stats(d)} for g, d in groups.items()}

def smd(a, b, k):
    (ma, va), (mb, vb) = a.get(k, (0, 0)), b.get(k, (0, 0))
    den = np.sqrt((va + vb) / 2)
    return 0.0 if den == 0 else (ma - mb) / den

keys_ = sorted(set().union(*[s.keys() for s in st.values()]))
bal = pd.DataFrame({
    "feature":    keys_,
    "smd_before": [smd(st["members_all"], st["pool_all"], k) for k in keys_],
    "smd_after":  [smd(st["members_kept"], st["lookalikes"], k) for k in keys_]}).round(3)
bal["pass"] = bal["smd_after"].abs() < 0.1
bal = bal.reindex(bal["smd_after"].abs().sort_values(ascending=False).index)

print(f"Features: {len(bal)} | Pass (|SMD| < 0.1): {bal['pass'].sum()} | Fail: {(~bal['pass']).sum()}")
display(spark.createDataFrame(bal))

## Step 6 — August read: KPIs and lift (Feb–Jul 2026)

**What this does**
Takes the matched members and their 5 lookalikes each, and measures what they actually did in the 6 months **after** matching (Feb 1 – Jul 31, 2026): visits, spend, concessions, tickets, premium. Lift = member value − lookalike value, with a 95% range showing how sure we are.

**So what**
Members and lookalikes looked alike on Jan 1. Any gap that opens up over Feb–Jul is the difference CineClub membership makes, beyond who the members already were. If a range doesn't cross 0, the lift is real, not noise.

Headline = members still active on Aug 1 (vs their own lookalikes). Side check = all matched members, including those who left.

In [0]:
KPI_START, KPI_END = 20260201, 20260801          # 6 months after Jan 1 matching
PREMIUM_EXP = [2, 3, 4, 7, 8]                     # AVX, IMAX, VIP, 4DX, ScreenX (official premium tickets)
COMP = ["visits", "movie_visits", "total_rev", "ticket_rev", "conc_rev",
        "visits_w_conc", "tickets", "prem_tickets", "used_premium"]

ids = (pairs.select(F.col("member_id").alias("CDE_ID"))
         .union(pairs.select(F.col("lookalike_id").alias("CDE_ID"))).distinct())
theatres = (spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")
              .filter(F.col("LC_Location_Type_Description") == "Cineplex Theatre").select(F.col("LocationID").alias("LocationId")).distinct())
sessions = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_session").select("SessionSkey", "Auditorium_Experience_ID")

tx = (spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
        .filter(F.col("VisitDateId").between(KPI_START, KPI_END - 1))
        .withColumn("CDE_ID", F.col("CDE_ID").cast("int"))
        .join(ids, "CDE_ID")
        .join(theatres, "LocationId")
        .join(sessions, "SessionSkey", "left"))

is_tkt = F.col("TicketFLAG") == "Yes"
visits = (tx.groupBy("CDE_ID", "VisitDateId", "LocationId")
            .agg(F.sum("Net_Revenue").alias("rev"),
                 F.sum(F.when(is_tkt, F.col("Net_Revenue")).otherwise(0)).alias("ticket_rev"),
                 F.sum(F.when(F.col("ConcessionFLAG") == "Yes", F.col("Net_Revenue")).otherwise(0)).alias("conc_rev"),
                 F.sum(F.when(is_tkt, F.col("Quantity")).otherwise(0)).alias("net_tickets"),
                 F.sum(F.when(is_tkt & F.col("Auditorium_Experience_ID").isin(PREMIUM_EXP), F.col("Quantity"))
                        .otherwise(0)).alias("prem_tickets"),
                 F.max(F.when(F.col("ConcessionFLAG") == "Yes", 1).otherwise(0)).alias("has_conc"))
            .withColumn("is_movie", (F.col("net_tickets") > 0) & (F.col("rev") > 0))
            .withColumn("is_visit", F.col("is_movie") | ((F.col("net_tickets") <= 0) & (F.col("has_conc") == 1) & (F.col("rev") > 0))))

person = (visits.groupBy("CDE_ID")
            .agg(F.count_if("is_visit").alias("visits"),
                 F.count_if("is_movie").alias("movie_visits"),
                 F.sum("rev").alias("total_rev"),
                 F.sum("ticket_rev").alias("ticket_rev"),
                 F.sum("conc_rev").alias("conc_rev"),
                 F.count_if(F.col("is_visit") & (F.col("has_conc") == 1)).alias("visits_w_conc"),
                 F.sum(F.when(F.col("is_movie"), F.col("net_tickets")).otherwise(0)).alias("tickets"),
                 F.sum(F.when(F.col("is_movie"), F.col("prem_tickets")).otherwise(0)).alias("prem_tickets"))
            .withColumn("used_premium", (F.col("prem_tickets") > 0).cast("int")))

person = ids.join(person, "CDE_ID", "left").fillna(0, COMP)          # no visits = zeros, still counted
P = person.toPandas().set_index("CDE_ID")[COMP].astype(float)
print(f"People with KPIs: {len(P):,} | with >= 1 visit in Feb-Jul: {(P['visits'] > 0).sum():,}")

In [0]:
pp   = pairs.select("member_id", "lookalike_id").toPandas()
look = P.reindex(pp["lookalike_id"].values).reset_index(drop=True)
look["member_id"] = pp["member_id"].values
look_avg = look.groupby("member_id")[COMP].mean()              # each member's 5 lookalikes averaged (reuse = weight)
mem      = P.reindex(look_avg.index)

active_aug = set(r["CDE_ID"] for r in spark.table(FEAT)
                 .filter((F.col("as_of_date") == "2026-08-01") & (F.col("cc_status") == "Active"))
                 .join(pairs.select(F.col("member_id").alias("CDE_ID")).distinct(), "CDE_ID")
                 .select("CDE_ID").collect())

def kpis(s, n):
    d = lambda a, b: a / b if b else np.nan
    return {
        "Avg frequency (visits / person)":     s["visits"] / n,
        "Total spend per visit ($)":           d(s["total_rev"], s["visits"]),
        "Ticket spend per movie visit ($)":    d(s["ticket_rev"], s["movie_visits"]),
        "Concession spend per visit ($)":      d(s["conc_rev"], s["visits"]),
        "Total spend per person, 6M ($)":      s["total_rev"] / n,
        "Ticket spend per person, 6M ($)":     s["ticket_rev"] / n,
        "Concession spend per person, 6M ($)": s["conc_rev"] / n,
        "Concession incidence (% of visits)":  100 * d(s["visits_w_conc"], s["visits"]),
        "Tickets per movie visit":             d(s["tickets"], s["movie_visits"]),
        "Premium share of tickets (%)":        100 * d(s["prem_tickets"], s["tickets"]),
        "People using premium (%)":            100 * s["used_premium"] / n}

def lift_table(idx, B=200, seed=42):
    M, Lk = mem.loc[idx][COMP].values, look_avg.loc[idx][COMP].values
    n = len(idx)
    km = kpis(dict(zip(COMP, M.sum(0))), n)
    kl = kpis(dict(zip(COMP, Lk.sum(0))), n)
    rng, boot = np.random.default_rng(seed), []
    for _ in range(B):                                          # resample members (with their lookalikes)
        w = rng.poisson(1.0, n).astype(float)
        bm = kpis(dict(zip(COMP, w @ M)), w.sum()); bl = kpis(dict(zip(COMP, w @ Lk)), w.sum())
        boot.append({k: bm[k] - bl[k] for k in km})
    boot = pd.DataFrame(boot)
    return pd.DataFrame({
        "KPI": list(km),
        "members": [km[k] for k in km],
        "lookalikes": [kl[k] for k in km],
        "lift": [km[k] - kl[k] for k in km],
        "lift_pct": [100 * (km[k] - kl[k]) / kl[k] if kl[k] else np.nan for k in km],
        "ci95_low": [boot[k].quantile(0.025) for k in km],
        "ci95_high": [boot[k].quantile(0.975) for k in km]}).round(2)

idx_active = [m for m in look_avg.index if m in active_aug]
print(f"Matched members: {len(look_avg):,} | Still active on Aug 1: {len(idx_active):,} "
      f"({100*len(idx_active)/len(look_avg):.1f}%)")

print("HEADLINE: members active on Aug 1 vs their lookalikes")
display(spark.createDataFrame(lift_table(idx_active)))
print("SIDE CHECK: all matched members (incl. leavers)")
display(spark.createDataFrame(lift_table(list(look_avg.index))))